# 🎛️ Deep Learning Studio · 1 of 5 — Classification: Who Is About to Leave?

**ISOM 839 · Prescriptive Analytics · Suffolk University · Prof. Hasan Arslan**

You will build your first neural network. It reads a customer's record and answers a yes/no question: **will this subscriber leave?** Then you use it to decide who gets a win-back offer.

**Classification** = the answer is a category (leave / stay, fraud / not fraud, approve / decline).

### 🧑‍🍳 The recipe (the same five steps in every Studio notebook)
| Step | Plain English |
|---|---|
| **1 · Numbers** | Turn the data into numbers |
| **2 · Shape** | Pick a network shape that fits the data |
| **3 · Train** | Let it practice: guess, measure the miss, nudge the knobs, repeat |
| **4 · Test** | Grade it on data it has never seen |
| **5 · Decide** | Use the prediction to make a business decision |

**Time:** about 20 minutes. Works with or without a GPU.

| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter**, no typing needed |
| 🤔 **Bet first** | say your guess out loud *before* you run the next cell |
| ✅ **What you should see** | if your output is close to this, move on (small differences are normal) |
| ✏️ **Your turn** | change one thing and re-run |

---
## Setup

In [ ]:
# @title ▶️ Run me: setup (about 20 seconds)
import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from IPython.display import display, HTML
keras.utils.set_random_seed(839)

class Progress(keras.callbacks.Callback):
    # prints one short line every few rounds of practice
    def __init__(self, every=5):
        super().__init__(); self.every = every
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f'round {epoch + 1:3d}:  ' + '   '.join(f'{k} {v:.3f}' for k, v in logs.items()))

def card(title, rows, color='#0a66c2'):
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:760px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready! Keras', keras.__version__)

**✅ What you should see:** `✅ Ready! Keras 3.…`

---
## Step 1 · Numbers — a customer becomes a row of numbers

A streaming service has 25,000 subscribers. We know 7 things about each, and whether they **left** last quarter. A hidden rule decides who leaves; the network never sees the rule, only examples.

In [ ]:
# @title ▶️ Run me: create the customers
def make_customers(n, seed=839):
    rng = np.random.default_rng(seed)
    tenure   = rng.integers(1, 73, n)
    contract = rng.choice(['month-to-month', 'one-year', 'two-year'], n, p=[0.55, 0.25, 0.20])
    charge   = np.round(rng.normal(70, 25, n).clip(20, 140), 2)
    calls    = rng.poisson(1.2, n)
    autopay  = rng.random(n) < 0.45
    usage    = np.round(rng.gamma(2.0, 6.0, n), 1)
    age      = rng.integers(18, 80, n)
    # 🙈 the hidden rule: the network has to discover it from examples
    z = (-1.2 + 1.4 * (contract == 'month-to-month') - 0.8 * (contract == 'two-year')
         - 1.6 * (1 - np.exp(-tenure / 12)) + 0.025 * (charge - 70) * (contract == 'month-to-month')
         + 0.8 * np.maximum(calls - 1, 0) - 0.6 * autopay + 0.9 * ((usage < 3) & (tenure < 12))
         + 0.0012 * (age - 45) ** 2 + 1.2 * ((charge > 95) & (calls >= 2)) - 0.3)
    p_true = 1 / (1 + np.exp(-z))
    left = (rng.random(n) < p_true).astype(int)
    df = pd.DataFrame(dict(tenure_months=tenure, contract=contract, monthly_charge=charge, support_calls=calls,
                           autopay=autopay.astype(int), data_gb=usage, age=age, left=left))
    return df, p_true

customers, p_true = make_customers(25_000)
display(customers.head())
print(f'{len(customers):,} customers. {customers.left.mean():.1%} of them left.')

**✅ What you should see:** 5 customers and `25,000 customers. 23.4% of them left.`

A network only understands numbers. Two small jobs:
1. **Words → numbers.** `contract` becomes three yes/no columns (this is called *one-hot encoding*).
2. **Same scale.** Age runs 18–80, autopay is 0 or 1. We rescale every column so none of them "shouts louder" (this is called *normalization*).

We also lock away the **last 5,000 customers as the exam**. The network never practices on them.

In [ ]:
# @title ▶️ Run me: words → numbers, and set the exam aside
X = pd.get_dummies(customers.drop(columns='left'), columns=['contract'], dtype='float32').astype('float32')
y = customers['left'].values.astype('float32')
columns = list(X.columns)

X_train, y_train = X.values[:20_000], y[:20_000]     # practice set
X_exam,  y_exam  = X.values[20_000:], y[20_000:]     # the exam: never used for practice

scaler = layers.Normalization()                      # learns each column's average and spread...
scaler.adapt(X_train)                                # ...from the practice set only

print('Customer #0 in words:  ', customers.iloc[0, :7].to_dict())
print('\nCustomer #0 as numbers:', X_train[0].round(1))
print('\nCustomer #0 rescaled:  ', np.array(scaler(X_train[:1]))[0].round(2))

**✅ What you should see:** the same customer three ways. The last line has small numbers around 0 (roughly −2 to +2). That row of 9 numbers is all the network will ever see of this person.

---
## Step 2 · Shape — start with ONE neuron

A **neuron** is a tiny scoring checklist:

> multiply each input by a **weight** (how much it matters) → add them up → squash the total into a number between 0 and 1.

The weights are **knobs**. Training means turning the knobs until the scores match what really happened.

**🤔 Bet first:** which column will get the biggest "pushes toward leaving" weight?

In [ ]:
# @title ▶️ Run me: build and train a one-neuron network
def knobs(model):
    return sum(int(np.prod(w.shape)) for w in model.trainable_weights)

one_neuron = keras.Sequential([
    keras.Input(shape=(9,)),                      # 9 numbers come in
    scaler,                                       # same scale
    layers.Dense(1, activation='sigmoid'),        # ONE neuron: weights, add up, squash to 0..1
])
one_neuron.compile(optimizer=keras.optimizers.Adam(0.01), loss='binary_crossentropy',
                   metrics=[keras.metrics.AUC(name='auc')])
print('Knobs to tune:', knobs(one_neuron))
one_neuron.fit(X_train, y_train, epochs=10, batch_size=256, verbose=0, callbacks=[Progress(5)])

weights = one_neuron.layers[-1].get_weights()[0][:, 0]
order = np.argsort(weights)
plt.figure(figsize=(8, 3.4))
plt.barh([columns[i] for i in order], weights[order], color=['#e2725b' if w > 0 else '#2bb673' for w in weights[order]])
plt.axvline(0, color='gray'); plt.title("The neuron's checklist: red pushes toward LEAVING, green toward STAYING"); plt.show()

**✅ What you should see:** `Knobs to tune: 10` (9 weights + 1 starting point called the *bias*), the `auc` climbing to about **0.78**, and a bar chart. **Support calls** and a **month-to-month contract** push toward leaving; a **long tenure** and a **two-year contract** push toward staying. It makes business sense!

> **AUC in one sentence:** pick one customer who left and one who stayed. AUC is how often the model says the leaver was riskier. 0.5 = coin flip, 1.0 = perfect.

One neuron can only draw a **straight line** through the data. It can't learn "expensive plan **and** lots of problems is especially bad." For that we need layers.

### Now a real network: neurons in layers

- **Layer 1 (32 neurons):** 32 different checklists look at the customer.
- **Layer 2 (16 neurons):** 16 checklists look at what layer 1 found. This is where **combinations** get noticed.
- **Output (1 neuron):** one final score between 0 and 1.

"**Deep** learning" just means: more than one layer.

In [ ]:
# @title ▶️ Run me: build the network
network = keras.Sequential([
    keras.Input(shape=(9,)),
    scaler,
    layers.Dense(32, activation='relu'),          # layer 1: 32 neurons
    layers.Dense(16, activation='relu'),          # layer 2: 16 neurons
    layers.Dense(1, activation='sigmoid'),        # output: chance of leaving, 0..1
])
network.compile(optimizer=keras.optimizers.Adam(0.003), loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='auc')])
print('Knobs to tune:', knobs(network))

**✅ What you should see:** `Knobs to tune: 865`. (ChatGPT-class models have hundreds of billions. Same idea, more knobs.)

`relu` is a simple switch inside each neuron: *pass the signal if it's positive, otherwise send 0.* That little switch is what lets layers learn bends and combinations instead of straight lines.

---
## Step 3 · Train — guess, measure the miss, nudge the knobs, repeat

- **Loss** = how wrong the guesses are (lower is better).
- **One round** (an *epoch*) = the network sees every practice customer once.
- We hold back 20% of the practice set as a **practice exam** (*validation*). When the practice-exam score stops improving, we stop. Otherwise the network starts **memorizing** instead of learning (that's called *overfitting*).

In [ ]:
# @title ▶️ Run me: train the network (about 30 seconds)
stop_early = keras.callbacks.EarlyStopping(patience=5, min_delta=0.0005, restore_best_weights=True)   # stop when the practice exam stops improving
history = network.fit(X_train, y_train, validation_split=0.2, epochs=60, batch_size=256,
                      verbose=0, callbacks=[Progress(5), stop_early])

plt.figure(figsize=(8, 3.4))
plt.plot(history.history['loss'], label='practice set'); plt.plot(history.history['val_loss'], label='practice exam')
plt.xlabel('round of practice (epoch)'); plt.ylabel('loss (how wrong)'); plt.legend(); plt.title('Watching it learn'); plt.show()
print('Stopped after', len(history.history['loss']), 'rounds')

**✅ What you should see:** both lines drop fast, then flatten. Training stops by itself after roughly **20 rounds** (yours may differ by a few). If the orange line starts creeping **up** while the blue keeps falling, that's memorizing; early stopping catches it.

---
## Step 4 · Test — the exam it has never seen

**🤔 Bet first:** how much better is 865 knobs than 10?

In [ ]:
# @title ▶️ Run me: grade both models on the 5,000 exam customers
from sklearn.metrics import roc_auc_score
p_one = one_neuron.predict(X_exam, verbose=0)[:, 0]
p_net = network.predict(X_exam, verbose=0)[:, 0]
crystal = roc_auc_score(y_exam, p_true[20_000:])
card('Exam results (AUC, higher is better)', [
    ('One neuron (10 knobs)', f'{roc_auc_score(y_exam, p_one):.3f}'),
    ('Network (865 knobs)', f'{roc_auc_score(y_exam, p_net):.3f}'),
    ('Perfect crystal ball (knows the hidden rule)', f'{crystal:.3f}')])

peek = customers.iloc[20_000:20_005, :7].copy(); peek['chance of leaving'] = (p_net[:5] * 100).round().astype(int).astype(str) + '%'; peek['really left?'] = y_exam[:5].astype(int)
display(peek)

**✅ What you should see:** one neuron about **0.78**, the network about **0.80**, and the crystal ball **0.814**. Even a perfect model can't reach 1.0, because people are partly random. The network closed most of the gap between the straight line and perfect.

---
## Step 5 · Decide — who gets the win-back offer?

A prediction is only useful if it changes a decision. The offer:

| | |
|---|---|
| Cost of sending an offer | **\$10** |
| If the customer was going to leave, the offer saves them | **30%** of the time |
| A saved customer is worth | **\$150** |

Sending to someone with leaving chance *p* is worth **p × 30% × \$150 − \$10**. That's positive only when **p is above 22%**. So the rule is: *send the offer if the model says the chance of leaving is above 22%.*

In [ ]:
# @title ▶️ Run me: profit of four ways to send offers (on the 5,000 exam customers)
OFFER_COST, SAVE_RATE, CUSTOMER_VALUE = 10, 0.30, 150      # ✏️ change these later
cutoff = OFFER_COST / (SAVE_RATE * CUSTOMER_VALUE)

def profit(send):                                          # send = True/False for each exam customer
    return float(np.sum(send * (y_exam * SAVE_RATE * CUSTOMER_VALUE - OFFER_COST)))

plans = {'Send to nobody': np.zeros(5000, bool), 'Send to everybody': np.ones(5000, bool),
         'One neuron picks': p_one > cutoff, 'Network picks': p_net > cutoff,
         'Crystal ball picks': p_true[20_000:] > cutoff}
print(f'Rule: send the offer when the chance of leaving is above {cutoff:.0%}\n')
for name, send in plans.items():
    print(f'{name:20s} offers sent: {send.sum():5,}   profit: ${profit(send):9,.0f}')

**✅ What you should see:** sending to everybody just **breaks even** (about \$0). Letting the network pick sends **fewer than half as many offers** and earns about **\$17,300**, close to the crystal ball's **\$18,200**. Even the one neuron earns about \$16,000. A better prediction became real money because it changed **who** gets the offer.

### ✏️ Your turn
1. Make the offer pricier: set `OFFER_COST = 20` and re-run the last cell. *(✅ The cutoff rises to 44%, far fewer offers go out, and "send to everybody" now **loses about \$50,000**.)*
2. Go back to "build the network", change `32` and `16` to `8` and `4`, and re-run from there. Does a smaller network do worse? *(✅ Barely: about 0.80 with only 121 knobs. Bigger isn't always better.)*

### 🧠 Check yourself
Answer in your head first, then click to open.

<details><summary><b>1. Why did we lock away 5,000 customers before training?</b></summary>

To grade the network on people it has **never seen**. A score on the practice set can't tell learning from memorizing.

</details>

<details><summary><b>2. One neuron scored about 0.78, the network about 0.80. What can layers learn that one neuron can't?</b></summary>

**Combinations and bends.** One neuron draws a straight line. Layers can learn "expensive plan *and* many support calls is especially risky."

</details>

<details><summary><b>3. Where does the 22% cutoff come from?</b></summary>

An offer costs \$10 and is worth 30% × \$150 = \$45 if the customer was going to leave. It pays when *chance of leaving* × \$45 is more than \$10, so when the chance is above 10 ÷ 45 = **22%**.

</details>

---
## 🎯 What you just did
| Step | Here |
|---|---|
| **Numbers** | one-hot for words, same scale for every column |
| **Shape** | `Dense` layers: 9 → 32 → 16 → 1, sigmoid at the end for a yes/no chance |
| **Train** | loss = binary cross-entropy, early stopping on a practice exam |
| **Test** | AUC on 5,000 customers it never saw |
| **Decide** | send the offer when the chance of leaving is above 22% |

**The frontier:** for tables, *foundation models* such as TabICL and TabPFN are pretrained networks that predict without any training on your data. Try the **Great Tabular Race** notebook on the Session 10 page.

**Next:** Studio 2 — the answer becomes a **number** instead of yes/no: how many croissants should the bakery bake tomorrow? 🥐